Import Libraries

In [20]:
import numpy as np
import pandas as pd
from collections import Counter
import pickle


In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/FabricDefect"
data_dir = f"{PROJECT_ROOT}/data"

if os.path.exists(data_dir):
    print(f"Contents of {data_dir}:")
    for item in os.listdir(data_dir):
        print(item)
else:
    print(f"Directory not found: {data_dir}")

Contents of /content/drive/MyDrive/FabricDefect/data:
raw
metadata.csv
train
val
test


Load Metadata and Create Small Feature Set

In [23]:
from PIL import Image

# Load metadata
PROJECT_ROOT = "/content/drive/MyDrive/FabricDefect"
metadata = pd.read_csv(f"{PROJECT_ROOT}/data/metadata.csv")

# Use small subset for testing
sample_df = metadata.sample(200, random_state=42)

# Function to extract simple features (mean RGB)
def extract_simple_features(filepath):
    img = Image.open(filepath).convert("RGB").resize((32,32))
    arr = np.array(img)
    mean_r = np.mean(arr[:,:,0])
    mean_g = np.mean(arr[:,:,1])
    mean_b = np.mean(arr[:,:,2])
    return np.array([mean_r, mean_g, mean_b])

# Extract features and labels
X = np.array([extract_simple_features(f) for f in sample_df['filepath']])
y = sample_df['label'].values

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (200, 3)
y shape: (200,)


Implement Gini IMpurity

In [24]:
def gini_impurity(y):
    counts = Counter(y)
    total = len(y)
    return 1.0 - sum((count/total)**2 for count in counts.values())


Testing

In [25]:
print(gini_impurity(["defect1","defect1","defect2"]))


0.4444444444444444


Implement Node Class

In [26]:
class Node:
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, *, value=None):
        self.feature_index = feature_index  # index of feature to split
        self.threshold = threshold          # threshold value
        self.left = left                    # left child node
        self.right = right                  # right child node
        self.value = value                  # value for leaf nodes


Split Function

In [27]:
def split_dataset(X, y, feature_index, threshold):
    left_idx = np.where(X[:, feature_index] <= threshold)[0]
    right_idx = np.where(X[:, feature_index] > threshold)[0]
    return X[left_idx], X[right_idx], y[left_idx], y[right_idx]


Find Best Split

In [28]:
def best_split(X, y):
    n_features = X.shape[1]
    best_gini = 1.0
    best_index = None
    best_threshold = None

    for feature_index in range(n_features):
        thresholds = np.unique(X[:, feature_index])
        for threshold in thresholds:
            _, _, y_left, y_right = split_dataset(X, y, feature_index, threshold)
            if len(y_left)==0 or len(y_right)==0:
                continue
            gini_left = gini_impurity(y_left)
            gini_right = gini_impurity(y_right)
            gini_total = (len(y_left)*gini_left + len(y_right)*gini_right)/len(y)

            if gini_total < best_gini:
                best_gini = gini_total
                best_index = feature_index
                best_threshold = threshold
    return best_index, best_threshold


Recursive Tree Building

In [29]:
def build_tree(X, y, max_depth=5, min_samples_split=2, depth=0):
    # If all labels are same, make leaf
    if len(set(y)) == 1:
        return Node(value=y[0])

    # If max depth reached or too few samples
    if depth >= max_depth or len(y) < min_samples_split:
        leaf_value = Counter(y).most_common(1)[0][0]
        return Node(value=leaf_value)

    # Find best split
    feature_index, threshold = best_split(X, y)
    if feature_index is None:
        leaf_value = Counter(y).most_common(1)[0][0]
        return Node(value=leaf_value)

    # Split dataset
    X_left, X_right, y_left, y_right = split_dataset(X, y, feature_index, threshold)

    # Build child nodes recursively
    left_child = build_tree(X_left, y_left, max_depth, min_samples_split, depth+1)
    right_child = build_tree(X_right, y_right, max_depth, min_samples_split, depth+1)

    return Node(feature_index, threshold, left_child, right_child)



Prediction Function

In [30]:
def predict_tree(node, x):
    if node.value is not None:
        return node.value
    if x[node.feature_index] <= node.threshold:
        return predict_tree(node.left, x)
    else:
        return predict_tree(node.right, x)

# Predict on dataset
def predict(node, X):
    return np.array([predict_tree(node, x) for x in X])


Train Tree on Sample Data

In [31]:
tree = build_tree(X, y, max_depth=5)
y_pred = predict(tree, X)

# Accuracy
accuracy = np.mean(y_pred == y)
print("Training Accuracy on sample subset:", accuracy)


Training Accuracy on sample subset: 0.92


STEP 10 — Save Tree to Drive

In [32]:
tree_path = f"{PROJECT_ROOT}/utils/decision_tree.pkl"
with open(tree_path, "wb") as f:
    pickle.dump(tree, f)

print("Decision tree saved at:", tree_path)


Decision tree saved at: /content/drive/MyDrive/FabricDefect/utils/decision_tree.pkl


 Feature Importance

In [33]:
def feature_importance(node, importance=None):
    if importance is None:
        importance = {}
    if node.value is None:
        importance[node.feature_index] = importance.get(node.feature_index,0)+1
        feature_importance(node.left, importance)
        feature_importance(node.right, importance)
    return importance

importance = feature_importance(tree)
print("Feature importance:", importance)


Feature importance: {1: 6, 0: 15}
